# Dow Structure Study v0.2

固定した全候補ユニバースを市場ごとの定点で観測し、確定スイング4状態と将来成績・モメンタム強度の関係を調べる探索的研究です。安値割れイベントに条件付けません。

**既存αエンジン・Research Runner・v0.1・原本スナップショットを変更せず、新しい市場データも取得しません。** 主分析は株価構造、相対価格構造は独立した副分析です。既存スクリーナー・発注・TTL・運用設定は実行しません。

固定Driveデータの実解析・Colab実機動作は合成データテストとは別です。操作・定義・検証範囲は `docs/dow_structure_study_v02.md` を参照してください。


## 1. 固定スナップショットを保存したGoogle Driveを接続

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2. 設定（編集はこのセル）

初期入力は2026年11月4日の固定スナップショットです。見つからない場合も新規価格取得で代替しません。

主仕様は事前固定：SPAN=5、2024年1月1日以降の最初の市場ベンチマーク有効日から21観測日ごと、位相0、主評価21（副評価5/63）、4特徴量の0.5SDカリパー。Phase IIは同日・市場内の生12M株価モメンタム上位10%かつプラスです。結果を見て閾値や位相を変更しないでください。


In [ ]:
REPO_URL = 'https://github.com/avalon19920629-create/Alpha-Engine-backtest.git'
CODE_REF = 'codex/dow-structure-study-v0-2'
EXPECTED_BASE_SHA = '59db288274608ed8f6b8753dcfbe790e37ffe5cb'
SNAPSHOT_DIR = '/content/drive/MyDrive/alpha_engine/research_runs/20261104T124443Z_residual100_n6_9b9d99b92dcf'
OUTPUT_ROOT = '/content/drive/MyDrive/alpha_engine/analysis/dow_structure_v02'
BOOTSTRAP_SAMPLES = 500
SEED = 20261104


## 3. 入力確認・分析コードのSHA固定・依存関係導入

毎回新しい `/content` ディレクトリへコードを取得します。既存チェックアウトへpull/resetしません。指定起点コミットの子孫であることと、起点から既存ファイルへの変更・削除・置換がないことを確認します。GitコードとPython依存関係の取得は、市場データの新規取得とは別です。


In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

snapshot = Path(SNAPSHOT_DIR).expanduser().resolve()
output = Path(OUTPUT_ROOT).expanduser().resolve()
if not snapshot.is_dir():
    raise FileNotFoundError(f'固定入力が見つかりません: {snapshot}。新規取得で代替しません。')
if output == snapshot or snapshot in output.parents:
    raise ValueError('分析出力先は固定スナップショットの外側にしてください。')
required = (
    'price_data/price_cache.parquet', 'price_data/benchmark_prices.parquet',
    'price_data/price_cache_metadata.json', 'price_data/price_data_quality.csv',
    'research_run_metadata.json', 'engine_outputs/score_components.csv',
)
for name in required:
    if not (snapshot / name).is_file():
        raise FileNotFoundError(f'固定入力が不足しています: {snapshot / name}')

REPO_DIR = Path(tempfile.mkdtemp(prefix='dow-structure-study-', dir='/content'))
subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)

def resolve_code_ref(ref):
    for candidate in (ref, f'refs/remotes/origin/{ref}'):
        result = subprocess.run(
            ['git', '-C', str(REPO_DIR), 'rev-parse', '--verify', '--end-of-options',
             f'{candidate}^{{commit}}'], text=True, capture_output=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'clone済み参照を解決できません: {ref}')

CODE_SHA = resolve_code_ref(CODE_REF)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', EXPECTED_BASE_SHA, CODE_SHA],
    check=True,
)
existing_changes = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'diff', '--name-status', '--diff-filter=CDMRTUXB',
     EXPECTED_BASE_SHA, CODE_SHA, '--'], text=True, capture_output=True, check=True,
)
if existing_changes.stdout.strip():
    raise RuntimeError('既存ファイルに変更があるため停止します:\n' + existing_changes.stdout)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_SHA], check=True)
print('Pinned analysis code:', CODE_REF, CODE_SHA)
print('Unchanged existing-code base:', EXPECTED_BASE_SHA)
print('Fixed snapshot:', snapshot)
print('Analysis output root:', output)
assert (REPO_DIR / 'dow_structure_study.py').is_file()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-dow-structure.txt')],
    check=True,
)


## 4. 定点観測・4状態・将来比較を実行

入力ハッシュを照合してから開始し、終了時にも原本不変を確認します。全候補を対象とし、最終選定6銘柄や最終日のランキングを過去へ適用しません。

別Pythonプロセスで起動し、結果通知は毎回新規作成します。import前後の失敗でも前回の成功フォルダを今回の結果として表示しません。


In [ ]:
import json

STUDY_RECEIPT = Path(tempfile.mkdtemp(
    prefix='dow-structure-receipt-', dir=str(REPO_DIR.parent),
)) / 'result.json'
run_config = {
    'snapshot_dir': str(snapshot), 'output_root': str(output),
    'bootstrap_samples': BOOTSTRAP_SAMPLES, 'seed': SEED,
}
child_code = r"""
import json
import sys
from pathlib import Path
from dow_structure_study import run_study

receipt = Path(sys.argv[2])
try:
    run_dir = run_study(**json.loads(sys.argv[1]))
except Exception as exc:
    receipt.write_text(json.dumps({
        'call_status': 'failed', 'run_dir': str(getattr(exc, 'run_dir', '') or ''),
        'error': str(exc),
    }, ensure_ascii=False), encoding='utf-8')
    raise
else:
    receipt.write_text(json.dumps({'call_status': 'returned', 'run_dir': str(run_dir)}), encoding='utf-8')
"""
result = subprocess.run(
    [sys.executable, '-c', child_code, json.dumps(run_config), str(STUDY_RECEIPT)],
    cwd=REPO_DIR,
)
receipt = json.loads(STUDY_RECEIPT.read_text()) if STUDY_RECEIPT.exists() else {
    'call_status': 'failed', 'run_dir': '',
    'error': '今回の結果通知がありません。上の実行エラーを確認してください。',
}
RUN_DIR = Path(receipt['run_dir']) if receipt.get('run_dir') else None
print('Process exit code:', result.returncode)
print('Study receipt:', receipt)
if result.returncode:
    print('解析は失敗しました。部分成果物を完成済み結果として扱わないでください。')


## 5. 実行状態・数値・標本減少・品質記録

Phase Iの株価構造／全候補群／21観測日後を主結果として読み、四状態を同じ基準で確認します。分類不能、比較不能、右側打切り、異常影響除外は別の理由です。株価と相対価格を独立標本のように合算しません。

Phase IIの無調整分布は上位10%群内部の記録で、遷移はその定点の上位群所属を起点とします。Phase IIの調整比較は、事前固定した全母集団マッチから上位10%の対象側を抜いたもので、対照は全母集団に残ります。上位群内部だけでマッチした検定とは呼びません。

銘柄クラスタ区間と評価日クラスタ区間は別々に読みます。区間を結合した較正済み二方向クラスタ推論や、有意性・因果効果を主張しません。

不確実性区間はH=21・全期間・符号全体・四状態・全体／生12M上位10%群に固定しています。副評価・年別・3M符号別の記述統計に同じ区間検証を実施したとは扱いません。

異常影響除外では過去253共通価格と直近2高値・2安値すべての判定窓を確認します。順位・上位群は除外前に固定し、除外された上位銘柄を次順位で補充しません。

感度分析の集計表はH=21・全期間・3M符号全体・全候補／生12M上位10%群・全状態・市場ALL/US/JP・二系列別に固定しています（`parameters.sensitivity_summary_scope`）。代替仕様の全5/21/63日後の個別結果は `sensitivity/<specification>_outcomes.csv` に残りますが、他ホライズンを感度集計表で要約したとは扱いません。主分析の全ホライズン・年別・符号別・参考群の集計は維持します。

`excluded_influence_count`（異常影響除外）は `right_censored_count`（将来不足）やその他の未観測件数と別々に確認してください。


In [ ]:
import pandas as pd
from IPython.display import JSON, Markdown, display

if RUN_DIR is None:
    raise RuntimeError('今回の保存先が不明です。前回の結果ではなく実行セルのエラーを確認してください。')
meta_path = RUN_DIR / 'study_metadata.json'
if meta_path.is_file():
    study_meta = json.loads(meta_path.read_text(encoding='utf-8'))
    display(JSON(study_meta))
    if study_meta.get('status') != 'completed' or study_meta.get('validation_status') != 'passed':
        print('未完了または検査失敗です。検証済み結果として扱わないでください。')
else:
    print('未作成:', meta_path)
report = RUN_DIR / 'study_report.md'
if report.is_file():
    display(Markdown(report.read_text(encoding='utf-8')))

priority = (
    'structure_outcome_summary.csv', 'structure_distribution.csv',
    'matched_structure_comparisons.csv', 'matching_balance.csv',
    'extreme_momentum_analysis.csv', 'uncertainty.csv', 'structure_agreement.csv',
    'state_durations.csv', 'daily_state_durations.csv', 'extreme_transitions.csv',
    'freshness_analysis.csv', 'freshness_outcomes.csv',
    'sensitivity_analysis.csv', 'observation_concentration.csv',
    'correction_log.csv', 'momentum_reproduction.csv', 'data_quality_warnings.csv',
)
paths = [RUN_DIR / name for name in priority if (RUN_DIR / name).is_file()]
paths += [path for path in sorted(RUN_DIR.glob('*.csv')) if path not in paths]
for path in paths:
    display(Markdown(f'**{path.name}（先頭20行）**'))
    try:
        display(pd.read_csv(path, nrows=20))
    except pd.errors.EmptyDataError:
        print('記録行なし（ファイルは空です）。')


## 6. 図と保存先

四状態構成比、定点間遷移、将来相対リターン分布、モメンタム強度帯別構成を確認します。分類不能や比較不能の多い状態を隠して、四状態のうち都合のよいものだけを評価しません。

同じ状態が定点で続いていても、間の日次状態がずっと同じだったことは保証しません。`state_durations.csv` は定点上の連続回数と両端のカレンダー日差です。鮮度・現在位置・位相変更は別の探索表として読み、主結果に条件を事後追加しません。

上位群の遷移は遷移元定点の所属を条件とし、次の定点で上位群から離れた例も保存します。次回も上位群だった例だけを抜き出した遷移ではありません。

補助表 `daily_state_durations.csv` は全共通有効日上のモデル状態の連続区間で、定点だけの継続表とは別です。古典的な真のトレンド期間とは呼びません。`freshness_outcomes.csv` は鮮度帯・位置別の将来平均／中央値・観測／打切り件数を示す探索表で、主比較の絞り込み条件に使いません。


In [ ]:
from IPython.display import SVG

if RUN_DIR is None:
    raise RuntimeError('今回の分析フォルダがありません。')
chart_paths = sorted((RUN_DIR / 'charts').glob('*.svg'))
print('SVG charts:', len(chart_paths))
for path in chart_paths:
    display(Markdown(f'**{path.name}**'))
    display(SVG(filename=str(path)))
print('Fixed source snapshot (unchanged):', snapshot)
print('This analysis run:', RUN_DIR)
print('Pinned analysis code SHA:', CODE_SHA)
print('Driveでstudy_metadata.json・study_report.md・数値CSV・chartsを確認してください。')
print('有利な状態・閾値の発見は成功条件ではなく、既存運用へ自動反映もしません。')
